# FCN + VGG16 - river plume segmentation

Training notebook for the **FCN + VGG16** model from the river-plume segmentation
study (Sentinel-2/Landsat RGB composites, binary plume masks).

Pipeline position:

1. `00_data_preparation.ipynb` builds the HDF5 datasets used below
   (`train_data_augmented_vgg_norm.h5`, `val_data_vgg_norm.h5`).
2. This notebook trains the model, evaluates it on the test split and exports
   per-image prediction masks for `13_ensemble.ipynb`.

Key facts: input 256x256x3, encoder-specific ImageNet preprocessing
(`tensorflow.keras.applications.vgg16.preprocess_input`),
loss = 0.6 Dice + 0.4 Focal, Adam, first 10 VGG16 layers frozen.
All hyperparameters are collected in the configuration cell below.

In [ ]:
import os

# Use legacy Keras 2 behaviour on TF >= 2.16. Must be set before TensorFlow
# is imported.
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import json
import math
import random

import h5py
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                     Dropout, UpSampling2D, Concatenate,
                                     GlobalAveragePooling2D, Reshape)
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import preprocess_input

# Fix random seeds for reproducibility. GPU-level nondeterminism still means
# repeated runs reproduce the metrics statistically rather than bit-exactly.
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 1. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - all hyperparameters of this model live in this cell.
# ============================================================================

MODEL_NAME = "fcn_vgg16"

# --- Data -------------------------------------------------------------------
TRAIN_HDF5 = "train_data_augmented_vgg_norm.h5"          # produced by 00_data_preparation.ipynb
VALID_HDF5 = "val_data_vgg_norm.h5"            # produced by 00_data_preparation.ipynb
TEST_IMG_DIR = "split_data/test/images"
TEST_MASK_DIR = "split_data/test/Masks"

INPUT_SHAPE = (256, 256, 3)        # network input size
TARGET_SIZE = (256, 256)           # inference resize (width, height)

# --- Training ---------------------------------------------------------------
BATCH_SIZE = 8
EPOCHS = 300                  # upper bound; EarlyStopping controls the actual length
EARLY_STOPPING_PATIENCE = 30
RLROP_FACTOR = 0.7      # ReduceLROnPlateau multiplier
RLROP_PATIENCE = 8
CLIPNORM = 0.5              # gradient norm clipping

# Learning rate is selected automatically with the LR Finder (see below).
LR_FINDER_START = 1e-6
LR_FINDER_END = 1e-2
LR_FINDER_STEPS = 50

# --- Loss -------------------------------------------------------------------
DICE_WEIGHT = 0.6                  # combined loss = 0.6 * Dice + 0.4 * Focal
FOCAL_WEIGHT = 0.4
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

# --- Inference / evaluation --------------------------------------------------
PRED_THRESHOLD = 0.5               # binarization threshold on [0, 1] probabilities

# --- Outputs ------------------------------------------------------------------
BEST_MODEL_PATH = f"{MODEL_NAME}_best.h5"
FINAL_MODEL_PATH = f"{MODEL_NAME}_final.h5"
TRAINING_CONFIG_PATH = f"{MODEL_NAME}_training_config.json"
PROBABILITY_EXPORT_DIR = f"probability_masks/test/fcn_with_vgg16"
BINARY_EXPORT_DIR = f"test_masks/fcn_with_vgg16"

## 2. Data generator

Streams normalized image/mask batches from the HDF5 files.

In [ ]:
class HDF5DataGenerator(tf.keras.utils.Sequence):
    """Keras Sequence that streams (image, mask) batches from an HDF5 file.

    The HDF5 file is produced by 00_data_preparation.ipynb and contains two
    datasets: "images" (N, H, W, 3) float32, already normalized with the
    encoder-specific preprocess_input, and "masks" (N, H, W, 1) float32 in {0, 1}.
    """

    def __init__(self, filename, batch_size=10, shuffle=True):
        self.filename = filename
        self.batch_size = batch_size
        self.shuffle = shuffle
        with h5py.File(self.filename, "r") as f:
            self.data_size = f["images"].shape[0]
        self.indexes = np.arange(self.data_size)
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __len__(self):
        return math.ceil(self.data_size / self.batch_size)

    def __getitem__(self, index):
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        with h5py.File(self.filename, "r") as f:
            # Read items one by one: h5py fancy indexing requires sorted
            # indices, which a shuffled Sequence does not guarantee.
            batch_images = np.array([f["images"][i] for i in batch_indexes])
            batch_masks = np.array([f["masks"][i] for i in batch_indexes])
        return batch_images, batch_masks

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

## 3. Model architecture

In [ ]:
from tensorflow.keras.applications import VGG16


def build_fcn_vgg16(input_shape=(256, 256, 3), num_classes=1):
    """FCN-style network with a VGG16 encoder for 256x256 inputs.

    Skip connections from every VGG block; each decoder stage upsamples x2,
    reduces channels in the skip branch, concatenates and applies a double
    convolution. The first 10 encoder layers are frozen.
    """
    base_model = VGG16(input_shape=input_shape, include_top=False, weights="imagenet")

    for layer in base_model.layers[:10]:
        layer.trainable = False

    skip_layers = {
        "block1_conv2": base_model.get_layer("block1_conv2").output,
        "block2_conv2": base_model.get_layer("block2_conv2").output,
        "block3_conv3": base_model.get_layer("block3_conv3").output,
        "block4_conv3": base_model.get_layer("block4_conv3").output,
        "block5_conv3": base_model.get_layer("block5_conv3").output,
    }

    def conv_block(x, filters, kernel_size=3, dropout_rate=0.05):
        """Conv + BatchNorm + ReLU + light Dropout."""
        x = Conv2D(filters, kernel_size, padding="same", kernel_initializer="he_normal")(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)
        if dropout_rate > 0:
            x = Dropout(dropout_rate)(x)
        return x

    def decoder_block(x, skip_connection, filters, dropout_rate=0.05):
        """Upsample x2, reduce skip channels, concatenate, double convolution."""
        x = UpSampling2D(2, interpolation="bilinear")(x)

        skip_reduced = Conv2D(filters // 4, 1, padding="same",
                              kernel_initializer="he_normal")(skip_connection)
        skip_reduced = BatchNormalization()(skip_reduced)
        skip_reduced = Activation("relu")(skip_reduced)

        x = Conv2D(filters, 1, padding="same", kernel_initializer="he_normal")(x)
        x = BatchNormalization()(x)
        x = Activation("relu")(x)

        x = Concatenate()([x, skip_reduced])

        x = conv_block(x, filters, kernel_size=3, dropout_rate=dropout_rate)
        x = conv_block(x, filters, kernel_size=3, dropout_rate=dropout_rate)
        return x

    x = skip_layers["block5_conv3"]  # (8, 8, 512)

    x = Conv2D(512, 1, padding="same", kernel_initializer="he_normal")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = decoder_block(x, skip_layers["block4_conv3"], 256, 0.05)  # -> (16, 16)
    x = decoder_block(x, skip_layers["block3_conv3"], 128, 0.05)  # -> (32, 32)
    x = decoder_block(x, skip_layers["block2_conv2"], 64, 0.05)   # -> (64, 64)
    x = decoder_block(x, skip_layers["block1_conv2"], 32, 0.05)   # -> (128, 128)

    # VGG16 block1 sits at full resolution, so the last decoder block already
    # outputs 256x256 and no extra upsampling is required.
    x = conv_block(x, 16, kernel_size=3, dropout_rate=0)
    x = Conv2D(num_classes, (1, 1), activation="sigmoid",
               kernel_initializer="he_normal")(x)

    return Model(inputs=base_model.input, outputs=x)

## 4. Loss and metrics

Combined Dice + Focal loss; Dice coefficient is monitored on the validation set.

In [ ]:
def dice_coefficient(y_true, y_pred, smooth=1e-7):
    """Batch-wise Dice coefficient.

    The Dice score is computed per sample and then averaged over the batch,
    which keeps the metric independent of the batch size.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    batch_size = tf.shape(y_true)[0]
    y_true_flat = tf.reshape(y_true, [batch_size, -1])
    y_pred_flat = tf.reshape(y_pred, [batch_size, -1])

    intersection = tf.reduce_sum(y_true_flat * y_pred_flat, axis=1)
    union = tf.reduce_sum(y_true_flat, axis=1) + tf.reduce_sum(y_pred_flat, axis=1)

    dice = (2.0 * intersection + smooth) / (union + smooth)
    return tf.reduce_mean(dice)


def dice_loss(y_true, y_pred, smooth=1e-7):
    """Dice loss = 1 - Dice coefficient."""
    return 1.0 - dice_coefficient(y_true, y_pred, smooth)


def binary_focal_loss(y_true, y_pred, alpha=0.25, gamma=2.0):
    """Binary Focal Loss (Lin et al., 2017).

    Down-weights easy examples and focuses training on hard pixels such as
    plume boundaries; alpha balances the positive class.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    epsilon = tf.keras.backend.epsilon()
    y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

    ce = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
    alpha_weight = y_true * alpha + (1 - y_true) * (1 - alpha)
    focal_weight = y_true * tf.pow(1 - y_pred, gamma) + (1 - y_true) * tf.pow(y_pred, gamma)

    focal = alpha_weight * focal_weight * ce
    return tf.reduce_mean(focal)


def combined_loss(y_true, y_pred, dice_weight=DICE_WEIGHT, focal_weight=FOCAL_WEIGHT):
    """Combined loss: DICE_WEIGHT * Dice loss + FOCAL_WEIGHT * Focal loss."""
    dice = dice_loss(y_true, y_pred)
    focal = binary_focal_loss(y_true, y_pred, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
    return dice_weight * dice + focal_weight * focal

## 5. Learning-rate selection (LR Finder)

In [ ]:
class SimpleLRFinder:
    """Learning Rate Finder (Smith, 2017).

    Sweeps the learning rate geometrically from LR_FINDER_START to
    LR_FINDER_END over LR_FINDER_STEPS single-batch updates, records the
    smoothed loss, and recommends the LR one order of magnitude below the
    loss minimum. Model weights are restored after the sweep.
    """

    def __init__(self, model, train_generator):
        self.model = model
        self.train_generator = train_generator
        self.lrs = []
        self.losses = []

    def find_lr(self, start_lr=1e-6, end_lr=1e-2, num_steps=50):
        print("Searching for the optimal learning rate...")

        original_weights = self.model.get_weights()

        lr_mult = (end_lr / start_lr) ** (1.0 / num_steps)
        current_lr = start_lr

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=current_lr),
            loss=combined_loss,
            metrics=["accuracy", dice_coefficient],
        )

        avg_loss = 0
        best_loss = 1e9

        for step in range(num_steps):
            try:
                batch_x, batch_y = next(iter(self.train_generator))
            except StopIteration:
                break

            K.set_value(self.model.optimizer.learning_rate, current_lr)

            logs = self.model.train_on_batch(batch_x, batch_y)
            loss = logs[0] if isinstance(logs, list) else logs

            # Exponential smoothing of the loss curve.
            avg_loss = 0.95 * avg_loss + 0.05 * loss
            smoothed_loss = avg_loss / (1 - 0.95 ** (step + 1))

            self.lrs.append(current_lr)
            self.losses.append(smoothed_loss)

            # Stop early if the loss explodes.
            if smoothed_loss > 3 * best_loss or np.isnan(smoothed_loss):
                print(f"Stopped at step {step}: loss diverged")
                break

            if smoothed_loss < best_loss:
                best_loss = smoothed_loss

            current_lr *= lr_mult

            if step % 10 == 0:
                print(f"Step {step}/{num_steps}, LR: {current_lr:.2e}, Loss: {smoothed_loss:.4f}")

        self.model.set_weights(original_weights)
        print("Search finished.")

        return self.recommend_lr()

    def recommend_lr(self):
        if len(self.losses) < 10:
            return 1e-4

        # Skip the first 5 noisy points, then take LR at the loss minimum
        # divided by 10.
        min_idx = np.argmin(self.losses[5:]) + 5
        optimal_lr = self.lrs[min_idx] / 10

        print(f"Recommended learning rate: {optimal_lr:.2e}")
        return optimal_lr

    def plot(self):
        if len(self.lrs) == 0:
            return

        plt.figure(figsize=(10, 6))
        plt.plot(self.lrs, self.losses, "b-", linewidth=2)
        plt.xlabel("Learning Rate")
        plt.ylabel("Loss")
        plt.title("Learning Rate vs Loss")
        plt.xscale("log")
        plt.grid(True, alpha=0.3)

        min_idx = np.argmin(self.losses[5:]) + 5 if len(self.losses) > 5 else 0
        if min_idx < len(self.lrs):
            plt.axvline(x=self.lrs[min_idx] / 10, color="r", linestyle="--",
                        label=f"Recommended: {self.lrs[min_idx] / 10:.2e}")
        plt.legend()
        plt.show()

## 6. Training callbacks

In [ ]:
def create_callbacks(model_path, monitor="val_dice_coefficient"):
    """Standard callback set: early stopping, LR schedule, best checkpoint.

    Settings are taken from the configuration cell above; the same protocol is
    used by every CNN model in the study.
    """
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor,
            patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            mode="max",
            verbose=1,
            min_delta=0.001,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            factor=RLROP_FACTOR,
            patience=RLROP_PATIENCE,
            min_lr=1e-7,
            mode="max",
            verbose=1,
            cooldown=3,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            model_path,
            monitor=monitor,
            save_best_only=True,
            mode="max",
            verbose=1,
            save_weights_only=False,
        ),
    ]


class TrainingMonitor(tf.keras.callbacks.Callback):
    """Prints train/val loss, Dice and the current LR every few epochs."""

    def __init__(self, display_freq=5):
        super().__init__()
        self.display_freq = display_freq
        self.best_dice = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        current_dice = logs.get("val_dice_coefficient", 0)

        if epoch % self.display_freq == 0:
            print(f"\nEpoch {epoch}:")
            print(f"  Train Loss: {logs.get('loss', 0):.6f} | Train Dice: {logs.get('dice_coefficient', 0):.6f}")
            print(f"  Val Loss:   {logs.get('val_loss', 0):.6f} | Val Dice:   {logs.get('val_dice_coefficient', 0):.6f}")
            print(f"  LR: {float(self.model.optimizer.learning_rate):.2e}")

            if current_dice > self.best_dice:
                self.best_dice = current_dice
                print(f"  New best Dice: {current_dice:.6f}")


class LearningRateSaver(tf.keras.callbacks.Callback):
    """Stores the final learning rate and optimizer settings in a JSON file."""

    def __init__(self, filepath="training_config.json"):
        super().__init__()
        self.filepath = filepath

    def on_train_end(self, logs=None):
        final_lr = float(self.model.optimizer.learning_rate)

        config = {
            "final_learning_rate": final_lr,
            "optimizer_config": {
                "learning_rate": final_lr,
                "beta_1": float(self.model.optimizer.beta_1),
                "beta_2": float(self.model.optimizer.beta_2),
                "epsilon": float(self.model.optimizer.epsilon),
            },
        }

        with open(self.filepath, "w") as f:
            json.dump(config, f, indent=4)

        print(f"Final learning rate saved: {final_lr:.2e}")
        print(f"Configuration written to: {self.filepath}")

## 7. Training

In [ ]:
def memory_optimization():
    """Enable GPU memory growth to avoid pre-allocating the whole GPU."""
    gpus = tf.config.experimental.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("GPU memory growth enabled")
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
    else:
        print("No GPU found, training on CPU")


memory_optimization()

print("Building the model...")
model = build_fcn_vgg16(input_shape=INPUT_SHAPE)
print(f"Model parameters: {model.count_params():,}")

train_generator = HDF5DataGenerator(TRAIN_HDF5, batch_size=BATCH_SIZE, shuffle=True)
valid_generator = HDF5DataGenerator(VALID_HDF5, batch_size=BATCH_SIZE, shuffle=False)

print("Running the Learning Rate Finder...")
lr_finder = SimpleLRFinder(model, train_generator)
learning_rate = lr_finder.find_lr(
    start_lr=LR_FINDER_START,
    end_lr=LR_FINDER_END,
    num_steps=LR_FINDER_STEPS,
)
lr_finder.plot()

print(f"Compiling the model with LR: {learning_rate:.2e}")
optimizer = tf.keras.optimizers.Adam(
    learning_rate=learning_rate,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7,
    clipnorm=CLIPNORM,
)

model.compile(
    optimizer=optimizer,
    loss=combined_loss,
    metrics=["accuracy", dice_coefficient],
)

callbacks = create_callbacks(BEST_MODEL_PATH)
callbacks.append(TrainingMonitor(display_freq=5))
callbacks.append(LearningRateSaver(TRAINING_CONFIG_PATH))

print("Starting training...")
history = model.fit(
    train_generator,
    validation_data=valid_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print("Training finished.")

best_val_dice = max(history.history.get("val_dice_coefficient", [0.0]))
print(f"Best validation Dice: {best_val_dice:.4f} "
      f"(enter this value for this model in 13_ensemble.ipynb)")

model.save(FINAL_MODEL_PATH)
print(f"Final model saved to: {FINAL_MODEL_PATH}")

# Training curves.
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["dice_coefficient"], label="Train Dice")
plt.plot(history.history["val_dice_coefficient"], label="Val Dice")
plt.title("Dice Coefficient")
plt.ylabel("Dice")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Test-set evaluation

Metrics: Dice, Accuracy, Precision, Recall and Hausdorff distances (HD95 / Average HD / Max HD) computed symmetrically with Euclidean distance transforms at the native ground-truth resolution.

In [ ]:
from PIL import Image
from scipy.ndimage import distance_transform_edt


def compute_dice(mask_true, mask_pred, eps=1e-7):
    """Dice coefficient between two binary masks (numpy)."""
    mask_true = mask_true.astype(np.float32)
    mask_pred = mask_pred.astype(np.float32)

    intersection = np.sum(mask_true * mask_pred)
    total = np.sum(mask_true) + np.sum(mask_pred)

    if total == 0:
        # Both masks empty: perfect agreement.
        return 1.0

    return (2.0 * intersection) / (total + eps)


def compute_metrics(mask_true, mask_pred, eps=1e-7):
    """Pixel-wise Accuracy, Precision and Recall for binary segmentation."""
    y_true = mask_true.flatten().astype(np.uint8)
    y_pred = mask_pred.flatten().astype(np.uint8)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / (total + eps)
    precision = 0.0 if (TP + FP) == 0 else TP / (TP + FP)
    recall = 0.0 if (TP + FN) == 0 else TP / (TP + FN)

    return accuracy, precision, recall


def compute_hausdorff_distances(mask_true, mask_pred):
    """HD95, Average HD and Max HD between two binary masks.

    Distances are computed symmetrically with Euclidean distance transforms
    over the full masks of an image.
    Returns (hd95, avg_hd, max_hd) in pixels.
    """
    true_coords = np.argwhere(mask_true > 0)
    pred_coords = np.argwhere(mask_pred > 0)

    if len(true_coords) == 0 or len(pred_coords) == 0:
        if len(true_coords) == 0 and len(pred_coords) == 0:
            return 0.0, 0.0, 0.0
        # One mask is empty: report the image diagonal as the distance.
        max_dist = np.sqrt(mask_true.shape[0] ** 2 + mask_true.shape[1] ** 2)
        return max_dist, max_dist, max_dist

    distances_pred_to_true = distance_transform_edt(1 - mask_true)
    distances_pred_to_true = distances_pred_to_true[pred_coords[:, 0], pred_coords[:, 1]]

    distances_true_to_pred = distance_transform_edt(1 - mask_pred)
    distances_true_to_pred = distances_true_to_pred[true_coords[:, 0], true_coords[:, 1]]

    all_distances = np.concatenate([distances_pred_to_true, distances_true_to_pred])

    hd95 = np.percentile(all_distances, 95)
    avg_hd = np.mean(all_distances)
    max_hd = np.max(all_distances)

    return hd95, avg_hd, max_hd


def binarize_mask(mask, threshold=0.5):
    """Binarize a mask with the given threshold."""
    return (mask > threshold).astype(np.uint8)


def find_corresponding_mask(img_name, mask_files):
    """Find the ground-truth mask file matching an image file name."""
    img_stem = os.path.splitext(img_name)[0]

    possible_names = [
        img_name,
        f"{img_stem}.png",
        f"{img_stem}.jpg",
        f"{img_stem}.jpeg",
        f"{img_stem}_mask.png",
        f"mask_{img_stem}.png",
        f"{img_stem}_gt.png",
        f"{img_stem}_groundtruth.png",
    ]

    for possible_name in possible_names:
        if possible_name in mask_files:
            return possible_name
    return None


def resize_to_match(mask, target_size):
    """Resize a binary mask to target_size with nearest-neighbour resampling."""
    if isinstance(mask, np.ndarray):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
    else:
        mask_pil = mask

    resized = mask_pil.resize(target_size, resample=Image.NEAREST)
    return (np.array(resized) > 127).astype(np.uint8)

In [ ]:
# Test-set evaluation. Metrics are computed at the native resolution of the
# ground-truth masks: the predicted 256x256 mask is upsampled back to the
# original image size before comparison.

img_names = sorted([f for f in os.listdir(TEST_IMG_DIR)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])
mask_files = sorted([f for f in os.listdir(TEST_MASK_DIR)
                     if f.lower().endswith((".png", ".jpg", ".jpeg"))])

print(f"Images found: {len(img_names)}")
print(f"Masks found:  {len(mask_files)}")

dice_scores, accuracy_scores = [], []
precision_scores, recall_scores = [], []
hd95_scores, avg_hd_scores, max_hd_scores = [], [], []

successful_count = 0
failed_count = 0

for i, img_name in enumerate(img_names):
    try:
        mask_name = find_corresponding_mask(img_name, mask_files)
        if mask_name is None:
            print(f"Mask not found for {img_name}, skipping")
            failed_count += 1
            continue

        img_path = os.path.join(TEST_IMG_DIR, img_name)
        mask_path = os.path.join(TEST_MASK_DIR, mask_name)

        orig_img = Image.open(img_path).convert("RGB")
        true_mask_img = Image.open(mask_path).convert("L")

        true_mask_np = np.array(true_mask_img)
        true_mask_bin = (true_mask_np > 127).astype(np.uint8)

        # Resize and apply the encoder-specific normalization.
        img_resized = orig_img.resize(TARGET_SIZE)
        img_np = np.array(img_resized).astype(np.float32)
        img_np = preprocess_input(img_np)
        img_input = np.expand_dims(img_np, axis=0)

        pred = model.predict(img_input, verbose=0)
        pred = np.squeeze(pred)

        # Binarize at PRED_THRESHOLD (0.5 on [0, 1] probabilities).
        pred_bin = binarize_mask(pred, threshold=PRED_THRESHOLD)

        # Upsample the prediction to the ground-truth resolution.
        pred_mask_resized = resize_to_match(pred_bin, true_mask_img.size)

        if true_mask_bin.shape != pred_mask_resized.shape:
            min_h = min(true_mask_bin.shape[0], pred_mask_resized.shape[0])
            min_w = min(true_mask_bin.shape[1], pred_mask_resized.shape[1])
            true_mask_bin = true_mask_bin[:min_h, :min_w]
            pred_mask_resized = pred_mask_resized[:min_h, :min_w]

        dice = compute_dice(true_mask_bin, pred_mask_resized)
        accuracy, precision, recall = compute_metrics(true_mask_bin, pred_mask_resized)
        hd95, avg_hd, max_hd = compute_hausdorff_distances(true_mask_bin, pred_mask_resized)

        dice_scores.append(dice)
        accuracy_scores.append(accuracy)
        precision_scores.append(precision)
        recall_scores.append(recall)
        hd95_scores.append(hd95)
        avg_hd_scores.append(avg_hd)
        max_hd_scores.append(max_hd)

        successful_count += 1
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(img_names)}")

    except Exception as e:
        print(f"Error while processing {img_name}: {e}")
        failed_count += 1

print("\n" + "=" * 70)
print("TEST-SET RESULTS: FCN + VGG16")
print("=" * 70)
print(f"Successfully processed: {successful_count}")
print(f"Failed:                 {failed_count}")

print(f"\nMean Dice Coefficient: {np.mean(dice_scores):.4f} +/- {np.std(dice_scores):.4f}")
print(f"Mean Accuracy:         {np.mean(accuracy_scores):.4f} +/- {np.std(accuracy_scores):.4f}")
print(f"Mean Precision:        {np.mean(precision_scores):.4f} +/- {np.std(precision_scores):.4f}")
print(f"Mean Recall:           {np.mean(recall_scores):.4f} +/- {np.std(recall_scores):.4f}")
print(f"\nHausdorff distances (pixels):")
print(f"Mean HD95:             {np.mean(hd95_scores):.4f} +/- {np.std(hd95_scores):.4f}")
print(f"Mean Average HD:       {np.mean(avg_hd_scores):.4f} +/- {np.std(avg_hd_scores):.4f}")
print(f"Mean Max HD:           {np.mean(max_hd_scores):.4f} +/- {np.std(max_hd_scores):.4f}")

## 9. Export predictions for the ensemble

In [ ]:
import cv2

# Export test-set predictions for the ensemble (13_ensemble.ipynb):
#   - probability masks: values in [0, 255] = probability * 255
#   - binary masks:      0 / 255 after thresholding at PRED_THRESHOLD
# File naming follows the convention <image_stem>_pred.png.

os.makedirs(PROBABILITY_EXPORT_DIR, exist_ok=True)
os.makedirs(BINARY_EXPORT_DIR, exist_ok=True)

img_names = sorted([f for f in os.listdir(TEST_IMG_DIR)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])

for img_name in img_names:
    img_path = os.path.join(TEST_IMG_DIR, img_name)
    orig_img = Image.open(img_path).convert("RGB")

    img_resized = orig_img.resize(TARGET_SIZE)
    img_np = np.array(img_resized).astype(np.float32)
    img_np = preprocess_input(img_np)
    img_input = np.expand_dims(img_np, axis=0)

    pred = np.squeeze(model.predict(img_input, verbose=0))

    img_stem = os.path.splitext(img_name)[0]

    prob_mask = (pred * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"), prob_mask)

    binary_mask = (binarize_mask(pred, threshold=PRED_THRESHOLD) * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(BINARY_EXPORT_DIR, f"{img_stem}_pred.png"), binary_mask)

print(f"Probability masks written to: {PROBABILITY_EXPORT_DIR}")
print(f"Binary masks written to:      {BINARY_EXPORT_DIR}")